In [16]:
import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')

from Data_Preparation.Embedding import embedding_encoder
from Data_Preparation.Tac import tac
from Data_Preparation import utils
from Modelisation.FlowMatching import flow_matching
from Modelisation.Baselines.OCSVM import ocsvm
from Modelisation.Baselines.CVDD.utils import build_vocab, cvdd_model_pipeline
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net


import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
import torch
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset, concatenate_datasets

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
%load_ext autoreload
%autoreload 2

In [20]:
train_20ng_, test_20ng_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
# train_reuters, test_reuters = utils.import_dataset(name="Reuters", batch_size=BATCH_SIZE)
# train_wos = utils.import_dataset(name="WOS", batch_size=BATCH_SIZE)
# train_dbpedia14, test_dbpedia14 = utils.import_dataset(name="DBpedia14", batch_size=BATCH_SIZE)
# train_agnews, test_agnews = utils.import_dataset(name="AGNews", batch_size=BATCH_SIZE)

20newsgroups dataset importing .... 




Repo card metadata block was not found. Setting CardData to empty.


In [29]:
inlier_topic = 'computer'
dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_20ng, train_anomaly_20ng = tac.textual_anomaly_contamination(train_20ng_.dataset, dataset_name, inlier_topic, type_tac, anomaly_rate, True)

print("TRAINSET")
print(train_inlier_20ng)
print(train_anomaly_20ng)

n_inliers_val = int(0.1 * len(train_inlier_20ng))
inlier_indices = np.random.choice(len(train_inlier_20ng), n_inliers_val, replace=False)
val_inlier_dataset = inlier_dataset.select(inlier_indices)

train_inlier_dataset = inlier_dataset.select([i for i in range(len(train_inlier_20ng)) if i not in inlier_indices])

n_anomalies_val = int(n_inliers_val / 0.9 * 0.1)
anomaly_indices = np.random.choice(len(train_anomaly_20ng), n_anomalies_val, replace=False)
val_anomaly_dataset = train_anomaly_20ng.select(anomaly_indices)

val_20ng = concatenate_datasets([val_inlier_dataset, val_anomaly_dataset]).shuffle(seed=42)
print("\nVALSET")
print(val_20ng)
print()

test_20ng = tac.textual_anomaly_contamination(test_20ng_.dataset, dataset_name, inlier_topic, type_tac, anomaly_rate, False)
print("TESTSET")
print(test_20ng)

TRAINSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 2862
})
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 318
})

VALSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 317
})

TESTSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 2121
})


In [30]:
model_name = 'all-MiniLM-L6-v2'

sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

model_name = 'distilbert-base-uncased'

bertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'bert')

model_name = 'glove_300d.kv'

gloveEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'glove')

In [31]:
train_inlier_20ng = bertEncoder.forward(train_inlier_20ng)
# train_anomaly_dl_20ng = sentencebertEncoder.forward(train_anomaly_dl_20ng)

test_20ng = bertEncoder.forward(test_20ng)
# test_dl_20ng = sentencebertEncoder.forward(test_dl_20ng)

val_20ng = bertEncoder.forward(val_20ng)
# val_20ng = sentencebertEncoder.forward(val_20ng

In [32]:
# X_inlier = Tensor(train_inlier_dl_20ng['sbert_embeddings']).to(device)
X_inlier = Tensor(train_inlier_20ng['bert_cls']).to(device)
# X_inlier = Tensor(train_inlier_dl_20ng['glove_embedding']).to(device)

X_inlier.shape

torch.Size([2862, 768])

## FM 

In [33]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
import numpy as np

# --- Dataset ---
batch_size_default = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size_default, shuffle=True)
input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Fonction d'objectif pour Optuna ---
def objective(trial):
    # Hyperparamètres à optimiser
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_epochs = trial.suggest_int("n_epochs", 100, 500, step=50)
    source = trial.suggest_categorical("source", ["gaussian", "sphere", "sphere-noised"])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-3)
    
    # DataLoader avec batch_size choisi
    dl_train = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)
    
    # Initialisation du modèle
    flow_model = flow_matching.FlowMatching(source, X_inlier.cpu(), input_dim, latent_dim, sinu, device).to(device)
    
    # Optimizer
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    
    # Trainer
    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=False)
    flow_model_trained = fm_trainer.train(dl_train, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    
    # Évaluation sur le val_set
    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)
    
    # Score combiné pour l'optimisation : on maximise AUC et AP, on minimise FPR95
    # Ici on peut pondérer les métriques
    score = auc + ap - fpr95
    return score

# --- Lancer l'étude Optuna ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # tu peux augmenter le nombre de trials

# --- Meilleurs hyperparamètres ---
print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)


[I 2025-11-25 18:37:19,476] A new study created in memory with name: no-name-4a9daf52-5630-4bb1-ab07-bbe7d8aee176
/tmp/ipykernel_3548261/3145040848.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
/tmp/ipykernel_3548261/3145040848.py:22: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-3)
[W 2025-11-25 18:37:40,462] Trial 0 failed with parameters: {'batch_size': 64, 'n_epochs': 500, 'source': 'sphere', 'lr': 7.514368141209551e-05, 'weight_decay': 2.349565192052122e-06} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  

KeyboardInterrupt: 

In [351]:
batch_size = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
lr = 1e-3
weight_decay = 1e-5
n_epochs = 200


target = X_inlier.cpu()
source = 'sphere-noised'
# source = trial.params['source']


flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

In [352]:
fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

In [353]:
flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

 step 0 -> loss : 0.08717
 step 40 -> loss : 0.06398
 step 80 -> loss : 0.06109
 step 120 -> loss : 0.06219
 step 160 -> loss : 0.06389


In [354]:
# X_test = torch.vstack([ Tensor(test_inlier_dl_20ng['sbert_embeddings']), Tensor(test_anomaly_dl_20ng['sbert_embeddings'])]).to(device)
# X_test = torch.vstack([ Tensor(test_inlier_dl_20ng['bert_cls']), Tensor(test_anomaly_dl_20ng['bert_cls'])]).to(device)

X_test =  Tensor(test_dl_20ng['bert_cls']).to(device)
y_test = np.array(test_dl_20ng['anomaly_class'])

In [355]:
auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

AUC: 0.9049 | FPR@95: 0.4468 | AP: 0.6390


## Basalines

### OCSVM

In [356]:
ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.8479 | FPR@95: 0.5427 | AP: 0.4066


### CVDD

In [14]:
type_emb = 'glove'
emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.01
lr_milestones = (5, 8)
n_epochs = 10
lambda_p = 1.0
alpha_scheduler = 'logarithmic'

In [15]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = train_inlier_dl_20ng['text']
    vocab = build_vocab(corpus,min_freq=1)
    tokenizer = None

In [16]:
cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_dl_20ng, test_dl_20ng, attention_size, n_attention_heads, 
                                               type_emb, 500, 64, True, device, tokenizer, vocab)

In [17]:
cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4, device=device)

model_trained = cvdd_trainer.train(cvdd_model, dl_train)

Starting training...
KMean starts
KMeans finish
| Epoch: 001/010 | Train Time: 0.517s | Train Loss: 0.139542 |
| Epoch: 002/010 | Train Time: 0.444s | Train Loss: 0.056700 |
| Epoch: 003/010 | Train Time: 0.441s | Train Loss: 0.049708 |
| Epoch: 004/010 | Train Time: 0.439s | Train Loss: 0.046049 |
| Epoch: 005/010 | Train Time: 0.441s | Train Loss: 0.042732 |
| Epoch: 006/010 | Train Time: 0.439s | Train Loss: 0.041651 |
| Epoch: 007/010 | Train Time: 0.439s | Train Loss: 0.040930 |
| Epoch: 008/010 | Train Time: 0.439s | Train Loss: 0.040041 |
| Epoch: 009/010 | Train Time: 0.439s | Train Loss: 0.039830 |
| Epoch: 010/010 | Train Time: 0.441s | Train Loss: 0.039731 |
Training Time: 5.272s
Finished training. 



In [18]:
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.5008 | FPR@95: 0.9654 | AP: 0.1162
